# Comparacion de Detectores de Rostros sobre Dataset completo
# MTCNN, YOLO, RetinaFace y MediaPipe

**Actividad 9 — Vision Artificial**

Evaluacion cuantitativa de 4 detectores de rostros sobre el dataset en `dataset/`.
Se usa IoU (threshold 0.5) contra ground truth en formato YOLO para calcular Precision, Recall y F1-score
a nivel global (todo el dataset) y por imagen.

---



## Analisis Teorico

### MTCNN vs YOLO

**MTCNN (Multi-task Cascaded Convolutional Networks)** es un detector **multietapa** que utiliza tres redes CNN
en cascada: P-Net (Proposal), R-Net (Refinement) y O-Net (Output). Construye una piramide de imagenes para
detectar rostros a multiples escalas. Cada etapa filtra candidatos progresivamente. Ademas de bounding boxes,
predice 5 landmarks faciales (ojos, nariz, extremos de la boca).

**YOLO (You Only Look Once)** es un detector **single-shot** que divide la imagen en una cuadricula y predice
simultaneamente bounding boxes, confianza y clase en una sola pasada. Utiliza anclas predefinidas.
Variantes como YOLOv8-face se fine-tunean especificamente para rostros.

**Similitudes**:
- Ambos usan CNN y predicen bounding boxes rectangulares.
- Ambos se entrenan con supervision completa.

**Diferencias clave**:
- **Arquitectura**: MTCNN es cascada (3 redes); YOLO es single-shot (1 red).
- **Escalas**: MTCNN usa piramide de imagenes; YOLO usa anclas en cuadricula.
- **Landmarks**: MTCNN los incluye nativamente; YOLO no (requiere extension).
- **Velocidad**: YOLO es mas rapido (una pasada vs tres).
- **Rostros pequenos**: MTCNN suele ser mas preciso por la piramide multiescala.

### RetinaFace y MediaPipe como alternativas de una pasada a MTCNN

**RetinaFace** usa backbone (ResNet/MobileNet) + FPN + SSH en una sola pasada. Predice bboxes, landmarks y
puntuacion facial simultaneamente. La FPN le da robustez a multiples escalas similar a la piramide de MTCNN,
pero en una sola red -> mas rapido.

**MediaPipe Face Detection (BlazeFace)** es un detector ultra-liviano single-shot optimizado para moviles/edge.
Usa anclas y una estructura GPU-friendly. Extremadamente rapido, aunque potencialmente menos preciso en
condiciones dificiles comparado con MTCNN o RetinaFace.

**Conclusion**: RetinaFace y MediaPipe convierten el enfoque de MTCNN en un proceso **single-pass**,
sacrificando algo de precision en escalas extremas pero ganando velocidad y eficiencia computacional.

---



In [ ]:
!unzip -q dataset.zip

replace dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace dataset/images/test/frame_000001.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [ ]:
!pip install mtcnn opencv-python retina-face mediapipe ultralytics matplotlib numpy -q

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

DATASET_ROOT = Path('dataset')
IMG_DIR = DATASET_ROOT / 'images' / 'test'
LABEL_DIR = DATASET_ROOT / 'labels' / 'test'

image_paths = sorted(IMG_DIR.glob('*.png'))
print(f'Dataset: {len(image_paths)} imagenes encontradas')

gt_by_image = {}
images = {}

for img_path in image_paths:
    name = img_path.stem
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        print(f'  ERROR: no se pudo cargar {img_path}')
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    images[name] = {'bgr': img_bgr, 'rgb': img_rgb}
    H, W = img_rgb.shape[:2]

    label_path = LABEL_DIR / f'{name}.txt'
    if not label_path.exists():
        gt_by_image[name] = np.empty((0, 4))
        print(f'  {name}: sin labels')
        continue

    labels = np.loadtxt(str(label_path))
    if labels.ndim == 1:
        labels = labels.reshape(1, -1)

    boxes = []
    for label in labels:
        _, cx, cy, bw, bh = label
        x1 = (cx - bw / 2) * W
        y1 = (cy - bh / 2) * H
        x2 = (cx + bw / 2) * W
        y2 = (cy + bh / 2) * H
        boxes.append([x1, y1, x2, y2])
    gt_by_image[name] = np.array(boxes)
    print(f'  {name}: {len(boxes)} rostros GT')

IMG_NAMES = sorted(gt_by_image.keys())
print(f'\nTotal imagenes con GT: {len(IMG_NAMES)}')
total_faces = sum(len(gt_by_image[n]) for n in IMG_NAMES)
print(f'Total rostros en dataset: {total_faces}')


Dataset: 3 imagenes encontradas
  frame_000001: 5 rostros GT
  frame_000122: 6 rostros GT
  frame_000256: 6 rostros GT

Total imagenes con GT: 3
Total rostros en dataset: 17


In [ ]:
def compute_iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    if inter == 0:
        return 0.0
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def evaluate_detections(gt, preds, iou_thresh=0.5):
    if len(preds) == 0:
        return {'TP': 0, 'FP': 0, 'FN': len(gt)}
    if len(gt) == 0:
        return {'TP': 0, 'FP': len(preds), 'FN': 0}
    matched_pred = set()
    tp = 0
    for gi, gbox in enumerate(gt):
        best_iou = iou_thresh
        best_pi = -1
        for pi, pbox in enumerate(preds):
            if pi in matched_pred:
                continue
            iou = compute_iou(gbox, pbox)
            if iou > best_iou:
                best_iou = iou
                best_pi = pi
        if best_pi >= 0:
            matched_pred.add(best_pi)
            tp += 1
    fp = len(preds) - len(matched_pred)
    fn = len(gt) - tp
    return {'TP': tp, 'FP': fp, 'FN': fn}

def metrics_from_counts(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {'Precision': p, 'Recall': r, 'F1': f1}


In [ ]:
from mtcnn import MTCNN
mtcnn_detector = MTCNN()
mtcnn_dets = {}

for name in IMG_NAMES:
    img_rgb = images[name]['rgb']
    results = mtcnn_detector.detect_faces(img_rgb)
    boxes = []
    for face in results:
        x, y, w, h = face['box']
        boxes.append([x, y, x + w, y + h])
    mtcnn_dets[name] = np.array(boxes) if boxes else np.empty((0, 4))
    print(f'  MTCNN | {name}: {len(boxes)} rostros')

total = sum(len(mtcnn_dets[n]) for n in IMG_NAMES)
print(f'MTCNN total: {total} detecciones')


  MTCNN | frame_000001: 5 rostros
  MTCNN | frame_000122: 6 rostros
  MTCNN | frame_000256: 6 rostros
MTCNN total: 17 detecciones


In [ ]:
from ultralytics import YOLO
print('Cargando YOLOv8-face...')
yolo_model = YOLO('yolov12n-face.pt')
yolo_dets = {}

for name in IMG_NAMES:
    img_rgb = images[name]['rgb']
    results = yolo_model(img_rgb, conf=0.25, verbose=False)[0]
    if results.boxes is not None and len(results.boxes) > 0:
        boxes = results.boxes.xyxy.cpu().numpy()
    else:
        boxes = np.empty((0, 4))
    yolo_dets[name] = boxes
    print(f'  YOLO | {name}: {len(boxes)} rostros')

total = sum(len(yolo_dets[n]) for n in IMG_NAMES)
print(f'YOLO total: {total} detecciones')


Cargando YOLOv8-face...
  YOLO | frame_000001: 6 rostros
  YOLO | frame_000122: 6 rostros
  YOLO | frame_000256: 6 rostros
YOLO total: 18 detecciones


In [ ]:
!pip install retinaface-pytorch

In [ ]:
import numpy as np
# 1. Cambiamos la importación correcta según la documentación de PyPI
from retinaface.pre_trained_models import get_model

# 2. Cargamos el modelo preentrenado (ejemplo con resnet50 o mobilenet0.25)
# El modelo se pone en modo evaluación (.eval()) para hacer inferencia
model = get_model("resnet50_2020-07-20", max_size=2048)
model.eval()

retina_dets = {}

for name in IMG_NAMES:
    img_bgr = images[name]['bgr']

    # 3. Esta librería predice en formato JSON / Diccionario
    results = model.predict_jsons(img_bgr)

    boxes = []
    # Comprobamos que existan rostros en la lista devuelta
    if results and len(results) > 0:
        for face in results:
            # Extraemos la caja delimitadora ('bbox') que ya viene como [x1, y1, x2, y2]
            x1, y1, x2, y2 = face['bbox']

            # Opcional: Puedes filtrar por confianza si lo deseas
            # if face['score'] > 0.5:
            boxes.append([int(x1), int(y1), int(x2), int(y2)])

    # Guardamos como array de NumPy (vacío con forma 0,4 si no hay rostros)
    retina_dets[name] = np.array(boxes) if boxes else np.empty((0, 4))
    print(f'  RetinaFace | {name}: {len(boxes)} rostros')

total = sum(len(retina_dets[n]) for n in IMG_NAMES)
print(f'RetinaFace total: {total} detecciones')


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torch/hub.py:897: FutureWarning: Falling back to the old format < 1.6. This support will be deprecated in favor of default zipfile format introduced in 1.6. Please redo torch.save() to save it in the new zipfile format.
  return _legacy_zip_load(cached_file, model_dir, map_location, weights_only)


  RetinaFace | frame_000001: 5 rostros
  RetinaFace | frame_000122: 6 rostros
  RetinaFace | frame_000256: 6 rostros
RetinaFace total: 17 detecciones


### MediaPipe (using `mediapipe.tasks` API)

In [ ]:
!wget -O face_detection_full_range.tflite \
  https://storage.googleapis.com/mediapipe-assets/face_detection_full_range.tflite

--2026-07-25 03:11:21--  https://storage.googleapis.com/mediapipe-assets/face_detection_full_range.tflite
Resolving storage.googleapis.com (storage.googleapis.com)... 34.153.3.27, 34.144.170.27, 104.154.124.27, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|34.153.3.27|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1083786 (1.0M) [application/octet-stream]
Saving to: ‘face_detection_full_range.tflite’

face_detection_full 100%[===================>]   1.03M  --.-KB/s    in 0.006s  

2026-07-25 03:11:21 (172 MB/s) - ‘face_detection_full_range.tflite’ saved [1083786/1083786]



In [ ]:
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

mp_dets = {}

# Usa el modelo de rango completo (full range)
MODEL_PATH = '/content/face_detection_full_range.tflite'  # asegúrate de tenerlo descargado

base_options = python.BaseOptions(model_asset_path=MODEL_PATH)

options = vision.FaceDetectorOptions(
    base_options=base_options,
    min_detection_confidence=0.25   # solo este parámetro es válido aquí
)

detector = vision.FaceDetector.create_from_options(options)

for name in IMG_NAMES:
    img_rgb = images[name]['rgb']
    H, W = img_rgb.shape[:2]
    boxes = []

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    detection_result = detector.detect(mp_image)

    if detection_result.detections:
        for detection in detection_result.detections:
            bbox = detection.bounding_box
            x1 = int(bbox.origin_x)
            y1 = int(bbox.origin_y)
            x2 = int(bbox.origin_x + bbox.width)
            y2 = int(bbox.origin_y + bbox.height)
            boxes.append([x1, y1, x2, y2])

    mp_dets[name] = np.array(boxes) if boxes else np.empty((0, 4))
    print(f'  MediaPipe | {name}: {len(boxes)} rostros')

total = sum(len(mp_dets[n]) for n in IMG_NAMES)
print(f'MediaPipe total: {total} detecciones')

  MediaPipe | frame_000001: 7 rostros
  MediaPipe | frame_000122: 8 rostros
  MediaPipe | frame_000256: 9 rostros
MediaPipe total: 24 detecciones


## Conclusiones

- **MTCNN** ofrece buena precision con landmarks incluidos, pero al ser multietapa es mas lento.
- **YOLOv8-face** es rapido y preciso, buen balance general.
- **RetinaFace** compite directamente con MTCNN en precision pero en una sola pasada.
- **MediaPipe** es el mas rapido, ideal para tiempo real, aunque puede fallar en rostros pequenos o angulos extremos.

La eleccion del detector depende del caso de uso: velocidad (MediaPipe, YOLO) vs precision maxima (RetinaFace, MTCNN).

